# FITTING EXAMPLE FOR THE USE OF  FSRQ_fit

Let's import the needed libraries for the code to run!

In [ ]:
from astropy.io import fits
from astropy import units as u
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,\
                                   AutoMinorLocator)
from astropy.visualization import quantity_support
from specutils import Spectrum1D
from specutils.spectra import SpectralRegion
from specutils.fitting import fit_lines, fit_generic_continuum, fit_continuum
from astropy.modeling.powerlaws import PowerLaw1D
from astropy.modeling import models, custom_model
from astropy.modeling.tabular import Tabular1D, Tabular2D
from astropy.modeling import Fittable1DModel, Parameter
import plotly.express as px
import plotly.graph_objs as go
from plotly.subplots import make_subplots
from plotly.offline import init_notebook_mode, iplot

# First let's load an spectrum

In [ ]:
%matplotlib inline
import glob

#Note that I converted the fits into a txt file previously
files = sorted(glob.glob('PKS1510-089/rest_spectra/*.fits_c.txt'))  # ruta local

file = pd.read_csv(files[0],names=['wavelength','flux'],delim_whitespace=True)


namefile = 'PKS1510_'+files[0][-17:-11]+'_EXAMPLE'

fig = px.line(file,x='wavelength',y='flux')

iplot(fig)

## Excluded and Included regions for the continuum fitting

These regions define the parts of the spectrum to include and/or exclude for the fitting of the continuum.

You can add more regions copying the same format as in the example.

In [ ]:
inclContRegions = [SpectralRegion(4020*u.AA,4080*u.AA),
                   SpectralRegion(4690*u.AA,4790*u.AA)] #In this example we'll not use these,
                                                        #but you can play with them



exclContRegions = [SpectralRegion(4000*u.AA,4028*u.AA),
                   SpectralRegion(4066*u.AA,4155*u.AA),
                   SpectralRegion(4170*u.AA,4202*u.AA),
                   SpectralRegion(4220*u.AA,4700*u.AA),
                   SpectralRegion(4800*u.AA,5460*u.AA)]

# Gaussians to fit to the lines

## Example of fitting H$ \beta $

In [ ]:
#### Parameters: initial values and bounds

gaussians = [
    # [Initial Value, Lower Limit, Upper Limit, Fixed]

    #### Gaussian 1 - Hbeta broad
    
    [[4871,4850,4910,False], #Center (Angstroms)
     [0.5,0.0,1,False], #Normalized Flux
     [3000,2500,10000,False]], #FWHM (km/s)
    
    #### Gaussian 2 - Hbeta narrow
    
    [[4861,4856,4866,False], #Angstroms
     [0.21,0.0,1.0,False],
     [700,500,1000,False]], #km/s
    
    #### Gaussian 3 - [OIII] doblete_1
    
    [[4959,4954,4964,False], #Angstroms
     [0.3,0.0,1.0,False],
     [700,500,1000,False]], #km/s
    
    #### Gaussian 4 - [OIII] doblete_2
    
    [[5007,5002,5012,False], #Angstroms
     [0.5,0.0,1.0,False],
     [700,500,1000,False]], #km/s
    
    #### Gaussian 5 - Absorption 1
    
    [[5067,5065,5100,False], #Angstroms
     [-0.2,-0.3,-0.1,False],
     [800,100,1500,False]], #km/s
    
    #### Gaussian 6 - Absorption 2
    
    [[5086,5065,5100,False], #Angstroms
     [-0.12,-0.3,-0.1,False],
     [800,100,1500,False]], #km/s
    
    #### Gaussian 7 - Hbeta very broad
    
    [[4871,4850,4910,False], #Angstroms
     [0.2,0.0,1.0,False],
     [6000,2500,10000,False]], #km/s
    
    #### Gaussian 8 - Absorption 3
    
    [[5121,5100,5150,False], #Angstroms
     [-0.1,-0.2,0.0,False],
     [800,100,1500,False]] #km/s
    
]

### Contraints for the [O III] lines

If the model is fitting the [OIII] lines, you have to pass indicate it with:

In [ ]:
hb_oiii = True #If this is True, please put Hbeta narrow, [OIII]4959 and 5007 in
               #the gaussians 2, 3 and 4, in that order.

### Fe II template

Select between the next templates for Fe II modeling:

'1435-2033 Angstroms', 
'2200-3400 Angstroms', 
'4000-7000 Angstroms',
'4000-5500 Angstroms (Multiple components)'

For this example, we'll use '4000-5500 Angstroms (Multiple components)'

In [ ]:
feii_temp = '4000-5500 Angstroms (Multiple components)'

The parameters for the Fe II template of multiple components in the range of 4000-5000 angstroms are specified as follows:

In [ ]:
#[init,min,max,fixed]

feii_params = [[[0.059,0.0,1.0,False], #Amplitude  #f component
                [-3.91,-20,20,False],  #Shift
                [2000,700,2800,False]],#Smooth of the template (in km/s from 700 to 2700 in steps of 100km/s)
                                      #ONLY CHANGE THE FIRST VALUE/USES SAME VALUE FOR ALL GROUPS    
        
               [[0.08,0.0,1.0,False], #Amplitude #g component
                [1.7,-20,20,False]],  #Shift
        
               [[0.068,0.0,1.0,False], #s component
                [13.9,-20,20,False]],
        
               [[0.054,0.0,1.0,True], #p component  
                [5.6,-20,20,True]],
        
               [[0.071,0.0,1.0,False], #Zw component
                [-15,-20,20,False]]]


#### For the case of templates for the range of Mg II and C IV the parameters are given
#### in the next way


#  feii_params = [[0.059,0.0,1.0,False], #Amplitude  #f component
#                [-3.91,-20,20,False],  #Shift
#                [10,1,50,True]]        #Smooth (from 1 to 50 in steps of 1) Larger number is smoother

# Fitting procedure

Finally, we have to setup the fitting window (spectral range in which perform the fitting) and pass the variables to the function.

In [ ]:
fit_window = SpectralRegion(4450*u.AA,5480*u.AA) #in Angstroms

from FSRQ_fit import Fit_Spec

final_model, params, cont_params = Fit_Spec(L=file.wavelength.values,FLUX=file.flux.values,gaussians=gaussians,\
                                            inclContRegions=None,exclContRegions=exclContRegions,\
                                            hb_oiii=hb_oiii,feii_temp=feii_temp,\
                                            feii_params=feii_params,fit_window=fit_window)

Here we manipulate the values of the obtained parameters to be easily accesible through a Python dictionary

In [ ]:
parameters = final_model.parameters

names = final_model.param_names

dic={}
for i in range(len(names)):
    dic[names[i]] = parameters[i]


# Now that we have a final model, let's visualize it!

### But first we have to assign the parameters to their respective models

In [ ]:
#Gaussian parameters
amplitudes = []
means = []
stddevs = []
for i in range(len(gaussians)):
    
    amplitudes.append(dic[f'amplitude_{i}'])
    means.append(dic[f'mean_{i}'])
    stddevs.append(dic[f'stddev_{i}'])
    
#Fe II parameters
amplituds = []
shifts = []
dopplers = []

for i in range(5):
    amplituds.append(dic[f'amplitud_{i+8}'])
    shifts.append(dic[f'shift_{i+8}'])
    dopplers.append(dic[f'doppler_{i+8}'])
    
    

In [ ]:
import FSRQ_fit

final_gaussians = []
    
#Assigning the gaussian parameters    
for i in range(len(gaussians)):
        
    
    g = models.Gaussian1D(amplitude=amplitudes[i],mean=means[i],\
                              stddev=stddevs[i])
        
    final_gaussians.append(g)
        
    if i == 0:
        all_models = g
    else:
        all_models = all_models + g
        
#Assigning the continuum parameters    
continuum = PowerLaw1D(amplitude=cont_params[0],x_0=cont_params[1],\
                            alpha=cont_params[2])
        
    
#Assigning the Fe II template parameters     
f = FSRQ_fit.modelHierro_f(amplituds[0],shifts[0],dopplers[0])
g = FSRQ_fit.modelHierro_g(amplituds[1],shifts[1],dopplers[1])
s = FSRQ_fit.modelHierro_s(amplituds[2],shifts[2],dopplers[2])
p = FSRQ_fit.modelHierro_p(amplituds[3],shifts[3],dopplers[3])
Zw = FSRQ_fit.modelHierro_Zw(amplituds[4],shifts[4],dopplers[4])

#Final model with the found parameters
all_models = all_models + continuum + f + g + s + p + Zw
hierro = f + g + s + p + Zw
        


### Finally! We can plot the results!

In [ ]:
FLUX = file.flux.values
L = file.wavelength.values

residual = file.flux.values-all_models(L)
    
df = pd.DataFrame({'lamb':L,'res':np.array(residual)})
    
fe_ = df[(df.lamb > 4800) & (df.lamb < 5300)]
    
rms_fe = np.std(fe_.res)
    
plt.close()
fig, axs = plt.subplots(2,1,sharex=True,gridspec_kw={'height_ratios':[3,1]})
        
fig.subplots_adjust(hspace=0)
    
axs[0].plot(L,FLUX,label='Spectrum')
axs[0].plot(L,all_models(L),label='Fit')
axs[0].plot(L,continuum(L),label='Continuum - Power-law')
axs[0].plot(L,hierro(L),label='Template Fe II')
    
gauss_names = [r'H$\beta$ Broad comp.',r'H$\beta$ Narrow comp.',\
                   r'[OIII] $\AA$4959',\
                   r'[OIII] $\AA$5007','Absorption 1', 'Absorption 2',\
                       r'H$\beta$ Very Broad comp.',\
                       'Absorption 3']
for i in range(len(gaussians)):
    axs[0].plot(L,final_gaussians[i](L),label=gauss_names[i])
        
# If you need to see the individual components of the template uncomments the lines below
# axs[0].plot(L,f(L))
# axs[0].plot(L,g(L))
# axs[0].plot(L,s(L))
# axs[0].plot(L,p(L))
# axs[0].plot(L,Zw(L))
axs[0].tick_params(which='both',labelsize=12,top=True,right=True,direction='in')
axs[0].xaxis.set_minor_locator(AutoMinorLocator())
axs[0].yaxis.set_minor_locator(AutoMinorLocator())
axs[0].set_ylabel('Normalized Flux',fontsize=12)
axs[0].legend()
    
axs[1].plot(L,residual)
axs[1].axhline(0,c='k',linestyle='--')
axs[1].tick_params(which='both',labelsize=12,top=True,right=True,direction='in')
axs[1].xaxis.set_minor_locator(AutoMinorLocator())
axs[1].yaxis.set_minor_locator(AutoMinorLocator())
axs[1].set_ylabel('Residual',fontsize=12)
axs[1].set_xlabel(r'Rest-Frame Wavelength ($\AA$)',fontsize=12)
# plt.xlim([4450,5480])
axs[1].set_ylim([-0.2,0.2])
    
fig.set_size_inches([9,11])
plt.savefig(namefile+'.pdf',bbox_inches='tight') #It also saves a pdf with this figure
plt.show()

    
    
    